# Model Preprocessing and Combination with Human Data

### Combine the Results of the models

In [1]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import glob

base_cols = ['experiment_label', 
                'sub', #subject identifier
                'condition', #condition (natural, feature, shape)
                'animacy', #animacy (only if animacy_inclusion is True)
                'filtered_acc', #accuracy after filtering
                ]
conditions = ['natural', 'feature', 'shape']
model_archs = ['convnext', 'clip_vit']

#animacies = ['anim_natural', 'inanim_natural', 'inanim_artificial']

familiar_model_summary_df = pd.DataFrame(columns=base_cols)
novel_model_summary_df = pd.DataFrame(columns=base_cols)
experiments = ['familiar', 'novel']

for experiment in experiments:

    print(f'Experiment: {experiment}_glint')
    for model_arch in model_archs:
        if experiment == 'familiar':
            file_path = f'/zpool/vladlab/active_drive/arsch/GLINT/glint_modeling/results_{experiment}/{model_arch}_KNN_all_conditions_9way_new.csv'
            animacies = ['anim_natural', 'inanim_natural', 'inanim_artificial']
            model = pd.read_csv(file_path)

        elif experiment == 'novel':
            file_path = f'/zpool/vladlab/active_drive/arsch/GLINT/glint_modeling/results_{experiment}/{model_arch}_KNN_all_conditions_4way_novel.csv'
            animacies = ['inanim_artificial']
            model = pd.read_csv(file_path)
            if "test_animacy" not in model.columns:
                model["test_animacy"] = "inanim_artificial"
                
        source_df = model

        model = pd.read_csv(file_path)
        print(f'Processing {model_arch}') 

        #Create a temporary data frame
        condition_summary = pd.DataFrame(columns=base_cols)
        
        #Loop through conditions
        for condition in conditions:
            #Loop through animacies
            for animacy in animacies:
                model_by_condition = source_df[
                    (source_df['condition'] == condition) & 
                    (source_df['test_animacy'] == animacy) 
                    #(combined_model_results_familiar['model'] == model_arch)] #if not extreme, we don't need to filter by stimulus duration, so we only filter by condition and animacy
                ]
                print(f'{condition} and {animacy}')
                # Calculate accuracy
                filtered_acc = model_by_condition['acc'].mean()
                print(f'Accuracy: {filtered_acc}')

                row = {
                'experiment_label': f'{experiment}_glint', #experiment and extreme status
                'sub': model_arch, #subject identifier
                'condition': condition, #condition (natural, scrambled, line_drawing)
                'animacy': animacy, #animacy (only if animacy_inclusion is True)
                'filtered_acc': filtered_acc, #accuracy after filtering
    
                }
                condition_summary = pd.concat([condition_summary, pd.DataFrame([row])], ignore_index=True) #append the row to the condition summary

        if experiment == 'familiar':
            familiar_model_summary_df = pd.concat([familiar_model_summary_df, condition_summary], ignore_index=True)
        elif experiment == 'novel':
            novel_model_summary_df = pd.concat([novel_model_summary_df, condition_summary], ignore_index=True)



Experiment: familiar_glint
Processing convnext
natural and anim_natural
Accuracy: 0.9833333333333333
natural and inanim_natural
Accuracy: 1.0
natural and inanim_artificial
Accuracy: 1.0
feature and anim_natural
Accuracy: 0.9833333333333333
feature and inanim_natural
Accuracy: 0.9
feature and inanim_artificial
Accuracy: 1.0
shape and anim_natural
Accuracy: 0.6333333333333333
shape and inanim_natural
Accuracy: 0.31666666666666665
shape and inanim_artificial
Accuracy: 0.6333333333333333
Processing clip_vit
natural and anim_natural
Accuracy: 1.0
natural and inanim_natural
Accuracy: 1.0
natural and inanim_artificial
Accuracy: 1.0
feature and anim_natural
Accuracy: 0.9
feature and inanim_natural
Accuracy: 0.9
feature and inanim_artificial
Accuracy: 0.9
shape and anim_natural
Accuracy: 0.8333333333333334
shape and inanim_natural
Accuracy: 0.43333333333333335
shape and inanim_artificial
Accuracy: 0.6333333333333333
Experiment: novel_glint
Processing convnext
natural and inanim_artificial
Accur